# Entraînement du composant `entity_linker` SpaCy natif

Cette section entraîne le composant **`entity_linker`** intégré de SpaCy à partir des données annotées (paires mention → person_id).

Dans le contexte du projet ANR DECIDON, il s'agit d'entrainer un modèle NEL pour resoudre les references à des personnes classées soit SPK soit PER inférées préalablement par un processus NER (sans correction manuelle) à l'appui d'une KB créée à partir de la base des données Sycomore documentant l'ensemble des députés et sénateurs notamment pour la III République.

A court terme, des améliorations sont prévues pour améliorer la resolution des personnes dont les fonctions peuvent être connues dans le contexte d'une séance de débat :
- ajout d'une étape adhoc entre le NER et le NEL pour resoudre les références associées à des fonctions de personnes présentes au sein d'une séance de débat parlementaire. Il s'agit des fonctions officielles parfois à longue durée (ministres) ou à courte durée (rapporteur ou président de séance)

Il est aussi prévu d'enrichir de manière iterative la KB avec les entités "émergeantes" sorties du processus NEL .


L'entity_linker fonctionne en deux étapes distinctes :

- Génération de candidats (get_candidates) : pour chaque span d'entité détecté (ex: "Brun"), spaCy interroge la KnowledgeBase pour récupérer la liste des entités possibles correspondant à ce texte exact — c'est une recherche par correspondance exacte de chaîne sur les alias enregistrés, pas de fuzzy matching, pas de recherche sémantique à ce stade.

- Désambiguïsation (le modèle entraîné) : parmi les candidats renvoyés à l'étape 1, le réseau de neurones (encodeur de contexte + comparaison de vecteurs) choisit lequel est le plus probable compte tenu du contexte de la phrase — c'est cette étape qu'on a entraînée.

TODO :
- brancher l'outil de désambiguisation intra-séance de Bertrand
- ameliorer le candidate generation (comportement par défaut: si le texte exact de la mention détectée par le NER n'est pas identique, caractère pour caractère, à un alias que vous avez enregistré dans la KB via kb.add_alias(...), aucun candidat n'est renvoyé — et l'entity_linker renvoie automatiquement NIL, sans même solliciter le réseau de désambiguïsation.)
- 

## 1. Installation des bibliothèques (hors env)

In [54]:
!pip install --upgrade ipywidgets #redemarrer kernel

## 2. Importation des bibliothèques

In [55]:
import json
import sqlite3
import os
import glob
import csv
from pathlib import Path
from typing import Optional
from dataclasses import dataclass, field

import numpy as np
import spacy
from spacy.kb import InMemoryLookupKB
from spacy.kb import KnowledgeBase, Candidate
from spacy.language import Language

import spacy
import spacy_lookups_data
print("spaCy:", spacy.__version__)
print("spacy-lookups-data:", spacy_lookups_data.__version__)

from tqdm import tqdm

print("✅ Imports OK")

spaCy: 3.8.16
spacy-lookups-data: 1.0.5
✅ Imports OK


In [56]:
import sys
print(sys.executable)

/home/cbrando/nel-env/bin/python3


In [57]:
import sys
!{sys.executable} -m pip install spacy-lookups-data --break-system-packages

In [58]:
import spacy_lookups_data
print(spacy_lookups_data.__file__)
print(spacy_lookups_data.__version__)

from spacy.lookups import load_lookups
lookups = load_lookups("fr", ["lemma_lookup", "lemma_rules", "lemma_exc", "lemma_index"])
print(lookups)

/home/cbrando/nel-env/lib/python3.13/site-packages/spacy_lookups_data/__init__.py
1.0.5


## 3. Configuration

In [59]:
# Chemin vers votre fichier SQLite
DB_PATH = "../data/prosopography.db" #"/content/drive/MyDrive/Colab Notebooks/data/DECIDON/BDProsopro/prosopography.db"   # ex: "/content/drive/MyDrive/ma_base.db"

# needed for getting integral text in page so entity spans remain valid
ALLDATA_NER_NEL = "../data/2026-08-05_export_label-studio_NER-NEL_pre-traite_AUGUST.json"

# ouput NEL model
NEL_MODEL_OUTPUT_DIR = "../models/"

# GT input with entity spans
GT_TABLE = "../data/2026-07-01-CHANDRA-GT-NEL-SPK_PER-linked_linked_consolidated_intra-seance - nel_consolidated_intra (26).csv"

In [60]:
# ──────────────────────────────────────────────
#  Schéma de la table SQLite "persons"
# ──────────────────────────────────────────────
SQL_TABLE          = "persons"
SQL_COL_ID         = "person_id"
SQL_COL_LAST_NAME  = "last_name"
SQL_COL_FIRST_NAME = "first_name"
SQL_COL_ALIAS      = "alias"
SQL_COL_BIRTH      = "birth_date"
SQL_COL_DEATH      = "death_date"
SQL_COL_WIKIDATA   = "wikidata_qid"
SQL_COL_WIKIPEDIA  = "wikipedia_url"
SQL_COL_SYCOMORE   = "sycomore_id"
SQL_COL_SENAT      = "senat_id"

# Séparateur utilisé dans la colonne 'alias' s'il y en a plusieurs.
# Mettez None si la colonne ne contient qu'un seul alias (jamais de séparateur).
ALIAS_SEPARATOR = "|"   # ex: "|", ",", ";" — à adapter selon vos données


# ──────────────────────────────────────────────
#  Filtre temporel pour la construction de la KB
# ──────────────────────────────────────────────

# Marge en années autour de SESSION_YEAR : une personne est incluse si elle
# est potentiellement vivante OU si elle a exercé un mandat dont la période
# chevauche [SESSION_YEAR - YEAR_MARGIN, SESSION_YEAR + YEAR_MARGIN]
YEAR_MARGIN = 3

# ──────────────────────────────────────────────
#  Schéma de la table SQLite "mandate"
#  → À adapter selon votre schéma réel
# ──────────────────────────────────────────────
SQL_MANDATE_TABLE        = "mandates"
SQL_MANDATE_COL_PERSON   = "person_id"
SQL_MANDATE_COL_ID       = "id"
SQL_MANDATE_COL_POSITION = "position"    # ex: "député de la Seine"
SQL_MANDATE_COL_ROLES    = "role"       # ex: "président de commission"
SQL_MANDATE_COL_START    = "start_date"
SQL_MANDATE_COL_END      = "end_date"

# ──────────────────────────────────────────────
#  Paramètres d'Entity Linking
# ──────────────────────────────────────────────
FUZZY_THRESHOLD  = 80    # score minimum RapidFuzz (0–100)
TOP_K_CANDIDATES = 5     # nb max de candidats à considérer

# Modèle SpaCy à utiliser
SPACY_MODEL = "fr_core_news_lg"

# classes NER à traiter
PER_LABELS = ["PER", "SPK"]

print("✅ Configuration chargée")

✅ Configuration chargée


## 4. Préparation des données d'entrainement

Quelques configuration sont possibles :
- **GT produit à partir d'un processus NER puis pseudo-NEL avec correction manuelle à chaque étape**
- GT produit à partir d'un processus NER + NEL (adapté) avec correction manuelle et une désambiguisation intra-séance (plus tard)

Il est necessaire de faire une jointure avec un export intégral LabelStudio pour recuperer le texte integral des pages afin de les associer aux données d'entrainement, en effet les spans des entités correspondent au texte de la page (et non d'un extrait comme dans le GT actuel).

In [61]:
import pandas as pd
import json
import re

# ============================================================
# 1. CHARGEMENT DU TABLEUR (Google Sheets)
# ============================================================
df = pd.read_csv(GT_TABLE, sep='\t')

# ============================================================
# 2. CHARGEMENT DU JSON (export Label Studio)
# ============================================================

with open(ALLDATA_NER_NEL, encoding="utf-8") as f:
    pages_data = json.load(f)

pages_by_task_id = {}
for task in pages_data:
    task_id = task.get("id")
    text = task.get("data", {}).get("text")
    if task_id is not None and text:
        pages_by_task_id[task_id] = text

print(f"{len(pages_by_task_id)} pages chargées depuis le JSON.")

# ============================================================
# 3. FILTRAGE DU GSHEET SUR LES CLASSES PERTINENTES (PER / SPK)
# ============================================================

print("Répartition des classes dans le gsheet :")
print(df["classe"].value_counts())

df_el = df[df["classe"].isin(["PER", "SPK"])].copy()
print(f"\n{len(df_el)} / {len(df)} lignes retenues après filtrage sur classe PER/SPK.")

# ============================================================
# 4. PARSING DE LA COLONNE "span" ("8223:8251" -> (8223, 8251))
# ============================================================

def parse_span(span_str: str):
    try:
        start_str, end_str = str(span_str).split(":")
        return int(start_str), int(end_str)
    except (ValueError, AttributeError):
        return None

df_el["span_parsed"] = df_el["span"].apply(parse_span)

# ============================================================
# 5. EXTRACTION DU QID WIKIDATA (ou NIL) DEPUIS "wikidata_url_valide"
# ============================================================

def extract_entity_id(value: str) -> str | None:
    value = str(value).strip()
    if value.upper() == "NIL":
        return "NIL"
    match = re.search(r"(Q\d+)", value)
    if match:
        return match.group(1)
    return None

df_el["entity_id"] = df_el["wikidata_url_valide"].apply(extract_entity_id)

# ============================================================
# Extraction d'un contexte plus petit focalisé sur l'annotation et pas la page entière
# ============================================================

def extract_context_window(full_text: str, start: int, end: int, n_chars_margin: int = 500):
    window_start = max(0, start - n_chars_margin)
    window_end = min(len(full_text), end + n_chars_margin)
    windowed_text = full_text[window_start:window_end]
    new_start = start - window_start
    new_end = end - window_start
    return windowed_text, new_start, new_end

# ============================================================
# 6. CONSTRUCTION DE TRAIN_EXAMPLES_RAW (+ TRAIN_EXAMPLES_TASK_IDS en parallèle)
# ============================================================

TRAIN_EXAMPLES_RAW = []
TRAIN_EXAMPLES_TASK_IDS = []   # <-- NOUVEAU : liste parallèle, même index que TRAIN_EXAMPLES_RAW
skipped = []
nil_count = 0

for _, row in df_el.iterrows():
    task_id = row["task_id"]
    entity_id = row["entity_id"]
    span = row["span_parsed"]
    mention_text = str(row.get("entity", "")).strip()

    if entity_id is None:
        skipped.append((task_id, row.get("wikidata_url_valide"), "valeur ni QID ni NIL dans wikidata_url_valide"))
        continue

    if span is None:
        skipped.append((task_id, row.get("span"), "span mal formé"))
        continue
    start, end = span

    full_text = pages_by_task_id.get(task_id)
    if full_text is None:
        skipped.append((task_id, span, "page introuvable dans le JSON (task_id absent)"))
        continue

    if end > len(full_text):
        skipped.append((task_id, span, f"span hors limites (texte long de {len(full_text)} caractères)"))
        continue

    windowed_text, new_start, new_end = extract_context_window(full_text, start, end, n_chars_margin=300)

    extracted = windowed_text[new_start:new_end]
    if mention_text and extracted.strip() != mention_text.strip():
        skipped.append((task_id, span, f"incohérence après fenêtrage: span='{extracted}' vs mention='{mention_text}'"))
        continue

    if entity_id == "NIL":
        nil_count += 1

    TRAIN_EXAMPLES_RAW.append(
        (windowed_text, {"links": {(new_start, new_end): {entity_id: 1.0}}})
    )
    TRAIN_EXAMPLES_TASK_IDS.append(task_id)   # <-- NOUVEAU : ajouté uniquement si la ligne est retenue

print(f"\n{len(TRAIN_EXAMPLES_RAW)} exemples construits ({nil_count} avec label NIL), {len(skipped)} lignes ignorées.")
assert len(TRAIN_EXAMPLES_RAW) == len(TRAIN_EXAMPLES_TASK_IDS), "Désynchronisation entre TRAIN_EXAMPLES_RAW et TRAIN_EXAMPLES_TASK_IDS !"

print("\ntrain exemple:")
for ex, tid in zip(TRAIN_EXAMPLES_RAW[:1], TRAIN_EXAMPLES_TASK_IDS[:1]):
    print(f"task_id={tid} | {ex}")

print("\nIgnored lines:")
for task_id, span, reason in skipped[:20]:
    print(f"  - [task_id={task_id}] span={span}: {reason}")

101 pages chargées depuis le JSON.
Répartition des classes dans le gsheet :
classe
PER    2877
SPK    1809
Name: count, dtype: int64


/tmp/ipykernel_1786/3839653732.py:8: DtypeWarning: Columns (0: persons_4_link_method, 1: persons_5_link_method, 2: persons_7_link_method, 3: persons_8_link_method, 4: persons_9_link_method, 5: persons_10_link_method, 6: persons_12_match_strategy, 7: persons_12_link_method, 8: persons_12_person_id, 9: persons_12_wikidata_url, 10: persons_12_first_name, 11: persons_12_last_name, 12: persons_13_match_strategy, 13: persons_13_link_method, 14: persons_13_person_id, 15: persons_13_wikidata_url, 16: persons_13_first_name, 17: persons_13_last_name) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(GT_TABLE, sep='\t')



4686 / 4686 lignes retenues après filtrage sur classe PER/SPK.

4678 exemples construits (172 avec label NIL), 8 lignes ignorées.

train exemple:
task_id=109 | (" d'office sur la commune de Saint-Cirq (Aveyron) une contribution extraordinaire; 3^rd^ Du projet de loi tendant à établir d'office une imposition extraordinaire sur la section de Rieutort, dépendant de la commune de Marchastel (Lozère). Communication d'une demande d'interpellation adressée, par M. Henri Brisson, à M. le ministre des affaires étrangères : MM. Henri Brisson, le ministre des affaires étrangères.\n\nDiscussion de l'interpellation de M. du Breuil de Saint-Germain sur les faits qui se sont passés à Vicq (arrondissement de Langres) le vendredi 13 juin : MM. du Breuil de Saint-Germain, le ministre ", {'links': {(300, 313): {'Q669725': 1.0}}})

Ignored lines:
  - [task_id=110] span=(6549, 6570): incohérence après fenêtrage: span='la sœur supé-

rieuse' vs mention='la sœur supé-  rieuse'
  - [task_id=112] span=(3004, 

#### Diagnostic : structure du gsheet


In [62]:
print(df.columns.tolist())
print()
print("Valeurs non-nulles par colonne contenant 'image_url' ou 'url':")
url_cols = [c for c in df.columns if "url" in c.lower()]
for col in url_cols:
    print(f"  {col}: {df[col].notna().sum()} / {len(df)} non-nulles")

print()
print("Répartition des labels (si une colonne de type existe, ex: 'label' ou 'category'):")
# Adapter le nom de la colonne selon votre gsheet
for candidate_col in ["label", "type", "category", "entity_type"]:
    if candidate_col in df.columns:
        print(df[candidate_col].value_counts())

print()
print("Exemple de lignes où per_fct_image_url est manquant:")
print(df[df["per_fct_image_url"].isna()].head(10))

print()
print("Exemple de lignes où per_fct_image_url est rempli:")
print(df[df["per_fct_image_url"].notna()].head(10))

['uuid', 'classe', 'entity', 'span', 'start_for_tri', 'contexte', 'task_id', 'annotation_id', 'seance', 'date', 'source', 'id_entity', 'check_validity', 'wikidata_url_valide', 'wikidata_url', 'vote', 'vote_result', 'per_fct_per_span', 'per_fct_fct_span', 'per_fct_image_url', 'per_fct_chamber', 'per_fct_date', 'per_fct_match_method', 'per_fct_match_score', 'persons_1_rank', 'persons_1_score', 'persons_1_match_strategy', 'persons_1_link_method', 'persons_1_link_score', 'persons_1_person_id', 'persons_1_wikidata_url', 'persons_1_first_name', 'persons_1_last_name', 'persons_1_position', 'persons_1_roles', 'persons_2_rank', 'persons_2_score', 'persons_2_match_strategy', 'persons_2_link_method', 'persons_2_link_score', 'persons_2_person_id', 'persons_2_wikidata_url', 'persons_2_first_name', 'persons_2_last_name', 'persons_2_position', 'persons_2_roles', 'persons_3_rank', 'persons_3_score', 'persons_3_match_strategy', 'persons_3_link_method', 'persons_3_link_score', 'persons_3_person_id', 'pe

In [63]:
# Vérification 1 : task_id est-il rempli pour toutes les lignes du gsheet ?
print("task_id manquants :", df["task_id"].isna().sum(), "/", len(df))
print("task_id uniques :", df["task_id"].nunique())

# Vérification 2 : structure du JSON - le champ "id" existe-t-il au niveau racine de chaque tâche ?
print(pages_data[0].keys())
print("id racine de la 1ère tâche :", pages_data[0].get("id"))

# Vérification 3 : cet id racine correspond-il bien au "task" vu dans les annotations imbriquées ?
first_task = pages_data[0]
nested_task_ref = first_task.get("annotations", [{}])[0].get("task")
print("id racine vs task imbriqué :", first_task.get("id"), "vs", nested_task_ref)

# Vérification 4 : combien de task_id du gsheet trouvent une correspondance dans le JSON ?
json_task_ids = {task.get("id") for task in pages_data}
gsheet_task_ids = set(df["task_id"].dropna().unique())
print("task_id du gsheet présents dans le JSON :", len(gsheet_task_ids & json_task_ids), "/", len(gsheet_task_ids))
print("task_id du gsheet ABSENTS du JSON :", gsheet_task_ids - json_task_ids)

task_id manquants : 0 / 4686
task_id uniques : 101
dict_keys(['id', 'annotations', 'file_upload', 'drafts', 'predictions', 'data', 'meta', 'created_at', 'updated_at', 'allow_skip', 'inner_id', 'total_annotations', 'cancelled_annotations', 'total_predictions', 'comment_count', 'unresolved_comment_count', 'last_comment_updated_at', 'project', 'updated_by', 'comment_authors'])
id racine de la 1ère tâche : 109
id racine vs task imbriqué : 109 vs 109
task_id du gsheet présents dans le JSON : 101 / 101
task_id du gsheet ABSENTS du JSON : set()


## 5. Chargement de la base de connaissances depuis SQLite

In [64]:
@dataclass
class PersonEntity:
    """Représente une entité personne issue de la table SQLite 'persons'."""
    entity_id: str
    last_name: str
    first_name: str = ""
    aliases: list[str] = field(default_factory=list)
    birth_date: Optional[str] = None
    death_date: Optional[str] = None
    internal_id: Optional[str] = None
    wikipedia_url: Optional[str] = None
    sycomore_id: Optional[str] = None
    senat_id: Optional[str] = None

    @property
    def name(self) -> str:
        """Nom canonique affiché : 'Prénom Nom' ou 'Nom' si pas de prénom."""
        if self.first_name:
            return f"{self.first_name} {self.last_name}".strip()
        return self.last_name

    @property
    def all_names(self) -> list[str]:
        """
        Toutes les chaînes pouvant désigner cette personne :
        - 'Prénom Nom'
        - 'Nom' seul (très fréquent dans les débats parlementaires)
        - tous les alias déclarés
        """
        names = set()
        last  = (self.last_name or "").strip()
        first = (self.first_name or "").strip()

        if last:
            names.add(last)
        if first and last:
            names.add(f"{first} {last}")
            
        for alias in self.aliases:
            alias = alias.strip()
            if alias:
                names.add(alias)

        return [n for n in names if n]

    @property
    def description(self) -> str:
        """Courte description lisible (dates + identifiants externes)."""
        parts = []
        if self.birth_date or self.death_date:
            parts.append(f"{self.birth_date or '?'} – {self.death_date or '?'}")
        if self.wikidata_qid:
            parts.append(f"Wikidata: {self.wikidata_qid}")
        if self.sycomore_id:
            parts.append(f"Sycomore: {self.sycomore_id}")
        if self.senat_id:
            parts.append(f"Sénat: {self.senat_id}")
        return " | ".join(parts)


def _split_aliases(raw: Optional[str]) -> list[str]:
    if not raw:
        return []
    if ALIAS_SEPARATOR is None:
        return [raw.strip()] if raw.strip() else []
    return [a.strip() for a in raw.split(ALIAS_SEPARATOR) if a.strip()]


def load_knowledge_base(
    db_path: str,
    year_min: Optional[int] = None,
    year_max: Optional[int] = None,
) -> list[PersonEntity]:
    """
    Charge les entités personnes depuis la table SQLite 'persons'.

    Si year_min/year_max sont fournis, ne charge que les personnes :
      - potentiellement vivantes pendant [year_min, year_max] :
        (naissance <= year_max) ET (décès inconnu OU décès >= year_min) ;
      - OU ayant exercé un mandat (table SQL_MANDATE_TABLE) dont la période
        chevauche [year_min, year_max].

    Les dates NULL sont traitées de façon permissive : une personne sans
    date de décès connue n'est jamais exclue par ce critère.
    """
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Base SQLite introuvable : {db_path}")

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()

    cur.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name=?",
        (SQL_TABLE,)
    )
    if not cur.fetchone():
        conn.close()
        raise ValueError(f"Table '{SQL_TABLE}' absente de la base. Vérifiez SQL_TABLE.")

    base_select = f"""
        SELECT {SQL_COL_ID}         AS person_id,
               {SQL_COL_LAST_NAME}  AS last_name,
               {SQL_COL_FIRST_NAME} AS first_name,
               {SQL_COL_ALIAS}      AS alias,
               {SQL_COL_BIRTH}      AS birth_date,
               {SQL_COL_DEATH}      AS death_date,
               {SQL_COL_WIKIDATA}   AS wikidata_qid,
               {SQL_COL_WIKIPEDIA}  AS wikipedia_url,
               {SQL_COL_SYCOMORE}   AS sycomore_id,
               {SQL_COL_SENAT}      AS senat_id
        FROM {SQL_TABLE}
    """

    params: list = []

    if year_min is None or year_max is None:
        # Pas de filtre temporel : comportement précédent (toute la base)
        query = base_select
    else:
        # La table des mandats existe-t-elle ?
        cur.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name=?",
            (SQL_MANDATE_TABLE,)
        )
        has_mandate_table = cur.fetchone() is not None

        # Condition 1 : "vivant pendant [year_min, year_max]"
        # NB: strftime('%Y', ...) suppose des dates au format ISO 'YYYY-MM-DD'.
        # Si vos dates sont stockées comme de simples entiers (années),
        # remplacez CAST(strftime('%Y', col) AS INTEGER) par CAST(col AS INTEGER).
        alive_condition = f"""
            (
                ({SQL_COL_BIRTH} IS NULL OR CAST(strftime('%Y', {SQL_COL_BIRTH}) AS INTEGER) <= ?)
                AND
                ({SQL_COL_DEATH} IS NULL OR CAST(strftime('%Y', {SQL_COL_DEATH}) AS INTEGER) >= ?)
            )
        """
        params += [year_max, year_min]

        if has_mandate_table:
            # Condition 2 : "a un mandat chevauchant [year_min, year_max]"
            mandate_condition = f"""
                OR {SQL_COL_ID} IN (
                    SELECT {SQL_MANDATE_COL_PERSON}
                    FROM {SQL_MANDATE_TABLE}
                    WHERE (
                        {SQL_MANDATE_COL_START} IS NULL
                        OR CAST(strftime('%Y', {SQL_MANDATE_COL_START}) AS INTEGER) <= ?
                    )
                    AND (
                        {SQL_MANDATE_COL_END} IS NULL
                        OR CAST(strftime('%Y', {SQL_MANDATE_COL_END}) AS INTEGER) >= ?
                    )
                )
            """
            params += [year_max, year_min]
        else:
            mandate_condition = ""
            print(f"⚠️  Table '{SQL_MANDATE_TABLE}' introuvable : filtre basé "
                  f"uniquement sur les dates de naissance/décès.")

        query = f"{base_select} WHERE {alive_condition} {mandate_condition}"

    cur.execute(query, params)
    rows = cur.fetchall()
    conn.close()

    entities = []
    for row in rows:
        entities.append(PersonEntity(
            entity_id     = row["wikidata_qid"],
            last_name     = row["last_name"] or "",
            first_name    = row["first_name"] or "",
            aliases       = _split_aliases(row["alias"]),
            birth_date    = row["birth_date"],
            death_date    = row["death_date"],
            internal_id  = row["person_id"],
            wikipedia_url = row["wikipedia_url"],
            sycomore_id   = row["sycomore_id"],
            senat_id      = row["senat_id"],
        ))

    return entities

def detect_year_range(json_dir: str, margin: int = YEAR_MARGIN):
    """
    Parcourt tous les fichiers JSON du dossier d'entrée et extrait
    l'année min et max trouvées dans le champ 'date' de chaque mention.
    Retourne (year_min, year_max) avec la marge appliquée,
    ou (None, None) si aucune date n'est trouvée.
    """
    years = []
    for json_path in glob.glob(os.path.join(json_dir, "*.json")):
        with open(json_path, encoding="utf-8") as f:
            mentions = json.load(f)
        if isinstance(mentions, dict):
            mentions = [mentions]
        for m in mentions:
            raw_date = m.get("date", "")
            if raw_date:
                try:
                    years.append(int(raw_date[:4]))
                except ValueError:
                    pass

    if not years:
        print("⚠️  Aucune date trouvée dans les fichiers JSON — filtre temporel désactivé.")
        return None, None

    year_min = min(years) - margin
    year_max = max(years) + margin
    print(f"📅 Plage temporelle détectée : {min(years)}–{max(years)} "
          f"(avec marge ±{margin} → filtre {year_min}–{year_max})")
    return year_min, year_max

# ── Appel avec filtre temporel ──────────────────────────────
#YEAR_MIN, YEAR_MAX = detect_year_range(JSON_INPUT_DIR)
kb_entities = load_knowledge_base(DB_PATH) #, year_min=YEAR_MIN, year_max=YEAR_MAX
print(f"✅ {len(kb_entities)} entités chargées depuis la table '{SQL_TABLE}' ")

print("\nExemples :")
for e in kb_entities[:5]:
    print(f"  [{e.entity_id}] {e.name}  | variantes : {e.all_names} ")

✅ 5053 entités chargées depuis la table 'persons' 

Exemples :
  [Q105906895] Antoine Blanc  | variantes : ['Antoine Blanc', 'Blanc'] 
  [Q2839785] Alphonse, Joseph Dupont  | variantes : ['Dupont', 'Alphonse, Joseph Dupont'] 
  [Q3407763] Prosper Blanc  | variantes : ['Prosper Blanc', 'Blanc'] 
  [Q327403] Henri, Antoine, Marie Germain  | variantes : ['Henri, Antoine, Marie Germain', 'Germain'] 
  [Q3426684] René Nicod  | variantes : ['René Nicod', 'Nicod'] 


**Le filtre temporel n'est plus d'actualité** car la BD Sqlite ne contiennent que des periodes pertinantes pour la 3e republique : Ce filtre temporelle est appliqué une fois pour construire la KB utilisée sur l'ensemble du corpus JSON. Si les sessions couvrent des années très différentes (par exemple 1880 à 1920), une fenêtre unique centrée sur 1903 exclurait à tort des personnes pertinentes pour les sessions plus tardives. Dans ce cas, le plus simple est de calculer YEAR_MIN/YEAR_MAX à partir du session_date minimum et maximum trouvés dans JSON_INPUT_DIR (plus la marge), plutôt que de fixer SESSION_YEAR en dur

## 6. Construction de la Knowledge Base SpaCy

In [65]:
# Chargement du modèle SpaCy
nlp = spacy.load(SPACY_MODEL)
print(f"✅ Modèle SpaCy chargé : {SPACY_MODEL}")

# Taille du vecteur d'entité (doit correspondre au modèle)
ENTITY_VECTOR_SIZE = nlp.vocab.vectors_length if nlp.vocab.vectors_length > 0 else 300
print(f"   Dimension des vecteurs : {ENTITY_VECTOR_SIZE}")

✅ Modèle SpaCy chargé : fr_core_news_lg
   Dimension des vecteurs : 300


In [66]:
def build_spacy_kb(
    nlp_model: Language,
    entities: list[PersonEntity],
    vector_size: int
) -> KnowledgeBase:
    """
    Construit une KnowledgeBase SpaCy à partir des entités SQLite.

    Pour chaque entité ayant un QID valide :
      - entity_vector = vecteur SpaCy du nom canonique ('Prénom Nom')
      - alias = toutes les variantes de nom (cf. PersonEntity.all_names)

    Les entités sans QID Wikidata sont exclues de la KB (impossible de les
    relier aux exemples d'entraînement, qui sont eux-mêmes basés sur les QID).
    """
    print("Construction de la KB SpaCy...")
    kb = InMemoryLookupKB(
        vocab=nlp_model.vocab,
        entity_vector_length=vector_size
    )

    n_skipped_no_qid = 0
    n_skipped_no_name = 0
    n_duplicate_qid = 0
    seen_qids = set()

    # Pré-agrégation des alias : un même alias (ex: "Brun") peut être
    # partagé par plusieurs entités (plusieurs personnes portant ce nom).
    # On regroupe donc alias -> liste de QID candidats, avec une probabilité
    # répartie également entre eux, plutôt que de silencieusement ignorer
    # les collisions (ce que faisait le try/except ValueError).
    alias_to_qids: dict[str, list[str]] = {}

    print("Ajout des entités dans la KB SpaCy...")
    for entity in tqdm(entities):
        qid = (entity.entity_id or "").strip()
        if not qid:
            n_skipped_no_qid += 1
            continue

        if not entity.name.strip():
            n_skipped_no_name += 1
            continue

        if qid in seen_qids: ## TODO : ce n'est plus pertinant car on a basculé aux ID de wikidata ****
            # Deux person_id différents partagent le même QID Wikidata :
            # on ne peut ajouter qu'une seule fois une entité donnée dans la KB.
            print("already seen qid: "+qid)
            n_duplicate_qid += 1
            continue
        seen_qids.add(qid)

        doc = nlp_model(entity.name)
        vec = doc.vector if doc.has_vector else np.zeros(vector_size, dtype="f")

        kb.add_entity(
            entity        = qid,
            freq          = 1.0,
            entity_vector = vec,
        )

        for alias_name in entity.all_names:
            alias_to_qids.setdefault(alias_name, []).append(qid)

    print("Ajout des alias dans la KB SpaCy...")
    n_alias_added = 0
    for alias_name, qids in alias_to_qids.items():
        # dédoublonnage au cas où un même QID apparaîtrait deux fois pour le même alias
        qids = list(dict.fromkeys(qids))
        proba = 1.0 / len(qids)
        try:
            kb.add_alias(
                alias         = alias_name,
                entities      = qids,
                probabilities = [proba] * len(qids),
            )
            n_alias_added += 1
        except ValueError as e:
            print(f"⚠️  Alias '{alias_name}' ignoré : {e}")

    if n_skipped_no_qid:
        print(f"⚠️  {n_skipped_no_qid} entité(s) ignorée(s) (pas de QID Wikidata)")
    if n_skipped_no_name:
        print(f"⚠️  {n_skipped_no_name} entité(s) ignorée(s) (nom vide)")
    if n_duplicate_qid:
        print(f"⚠️  {n_duplicate_qid} entité(s) ignorée(s) (QID Wikidata dupliqué)")

    print(f"✅ KB construite : {kb.get_size_entities()} entités, {n_alias_added} alias")
    return kb


spacy_kb = build_spacy_kb(nlp, kb_entities, ENTITY_VECTOR_SIZE)

Construction de la KB SpaCy...
Ajout des entités dans la KB SpaCy...


 22%|████████████████▊                                                            | 1102/5053 [00:03<00:10, 363.00it/s]

already seen qid: Q3085581


 25%|███████████████████                                                          | 1249/5053 [00:03<00:10, 346.96it/s]

already seen qid: Q2831506


...

already seen qid: Q3592145


 75%|█████████████████████████████████████████████████████████▉                   | 3798/5053 [00:10<00:03, 363.65it/s]

already seen qid: Q3263438
already seen qid: Q2835322


100%|█████████████████████████████████████████████████████████████████████████████| 5053/5053 [00:14<00:00, 348.97it/s]

Ajout des alias dans la KB SpaCy...
⚠️  9 entité(s) ignorée(s) (QID Wikidata dupliqué)
✅ KB construite : 5044 entités, 9177 alias


## 7. Validation des exemples pour l'entrainement

In [67]:
from spacy.training import Example
import random
from spacy.util import minibatch
from tqdm.notebook import tqdm

nlp_train = spacy.load(SPACY_MODEL, exclude=["entity_linker", "lemmatizer"])
entity_linker = nlp_train.add_pipe("entity_linker", last=True)
entity_linker.set_kb(lambda vocab: spacy_kb)

# ============================================================
# VALIDATION DES EXEMPLES AVANT ENTRAINEMENT
# ============================================================

def validate_and_build_example(nlp_model, text, links_annot, label="PER"):
    doc = nlp_model.make_doc(text)
    entities_spans = []
    for (start, end), entity_dict in links_annot.items():
        for entity_id in entity_dict.keys():
            if not isinstance(entity_id, str) or not entity_id.strip():
                return None, f"entity_id invalide (pas une string) : {entity_id!r} (type={type(entity_id)})"
        span = doc.char_span(start, end, alignment_mode="strict")
        if span is None:
            return None, f"span ({start},{end}) non alignable sur les tokens : '{text[start:end]!r}'"
        entities_spans.append((span.start_char, span.end_char, label))
    annot = {"entities": entities_spans, "links": links_annot}
    try:
        example = Example.from_dict(doc, annot)
    except ValueError as e:
        return None, f"Example.from_dict a échoué : {e}"
    return example, None


train_examples = []   # <-- CETTE liste, filtrée, est celle utilisée pour l'entraînement
train_task_ids = []   # <-- NOUVEAU : liste parallèle, même index que train_examples
rejected = []

for (text, annot), task_id in zip(TRAIN_EXAMPLES_RAW, TRAIN_EXAMPLES_TASK_IDS):
    example, error = validate_and_build_example(nlp_train, text, annot["links"], label="PER")
    if example is None:
        rejected.append((text[:60], annot["links"], error))
        continue
    train_examples.append(example)
    train_task_ids.append(task_id)   # <-- NOUVEAU : ajouté uniquement si l'exemple est retenu

print(f"{len(train_examples)} exemples valides, {len(rejected)} rejetés.")
assert len(train_examples) == len(train_task_ids), "Désynchronisation entre train_examples et train_task_ids !"

if rejected:
    print("Exemples rejetés :")
    for text_preview, links, error in rejected:
        print(f"   {links} → {error}")

4675 exemples valides, 3 rejetés.
Exemples rejetés :
   {(300, 308): {nan: 1.0}} → entity_id invalide (pas une string) : nan (type=<class 'float'>)
   {(300, 332): {'Q3085590': 1.0}} → span (300,332) non alignable sur les tokens : ''le ministre des colonies et de M''
   {(300, 307): {nan: 1.0}} → entity_id invalide (pas une string) : nan (type=<class 'float'>)


## 8. Entraînement du entity_linker natif SpaCy

Vérification de la durée de l'entrainement sur un petit exemple.

In [68]:
**
N_ITER_TEST = 2
sample_examples = train_examples[:50]
BATCH_SIZE = 32  # commencez prudent, vous pourrez augmenter si la RAM le permet
optimizer = nlp_train.initialize(get_examples=lambda: train_examples)

import time
t0 = time.time()
for i in range(N_ITER_TEST):
    losses = {}
    for batch in minibatch(sample_examples, size=BATCH_SIZE):
        nlp_train.update(batch, sgd=optimizer, losses=losses)
print(f"Temps pour {N_ITER_TEST} époques sur {len(sample_examples)} exemples : {time.time()-t0:.1f}s")

SyntaxError: invalid syntax (3833724456.py, line 1)

batch size = 16 -> Temps pour 2 époques sur 50 exemples : 3.2s
batch size = 32 -> Temps pour 2 époques sur 50 exemples : 2.0s


In [ ]:
# ============================================================
# ENTRAINEMENT — utilise train_examples (filtré), pas TRAIN_EXAMPLES_RAW
# ============================================================

from spacy.util import minibatch
from tqdm.notebook import tqdm
import time

N_ITER = 30
BATCH_SIZE = 32

t_start = time.time()
for i in range(N_ITER):
    random.shuffle(train_examples)
    losses = {}
    batches = list(minibatch(train_examples, size=BATCH_SIZE))
    for batch in tqdm(batches, desc=f"Époque {i+1}/{N_ITER}", leave=False):
        nlp_train.update(batch, sgd=optimizer, losses=losses)
    elapsed = time.time() - t_start
    print(f"Iter {i+1:3d}/{N_ITER} | loss={losses.get('entity_linker', 0):.4f} | temps écoulé: {elapsed/60:.1f} min")

nlp_train.to_disk(NEL_MODEL_OUTPUT_DIR+"model_18082026_SpacyEL_GT-NER-NEL")
print("✅ Modèle entity_linker sauvegardé dans : model_082026_SpacyEL_GT-NER-NEL/")

Iter=50, batch_size=16, char_window = 500 -> 300 minutes
Iter=30, batch_size=32, char_windows = 300 -> 927 minutes (trop long car ordi en veille ?)

## 9. Test du modèle sur 1 exemple donné

In [ ]:
import spacy
from spacy.tokens import Span

MODEL_PATH = NEL_MODEL_OUTPUT_DIR + "model_18082026_SpacyEL_GT-NER-NEL"
nlp_test = spacy.load(MODEL_PATH)

print(f"✅ Modèle chargé depuis : {MODEL_PATH}")
print(f"   Composants du pipeline : {nlp_test.pipe_names}")

test_text = "Pour assister M. le ministre de l'intérieur : M. Brun, conseiller d'Etat."

# On désactive temporairement le NER pour forcer notre propre span,
# mais on garde tout le reste du pipeline (parser/senter/tok2vec...)
# actif pour que les frontières de phrases soient bien calculées
# avant l'entity_linker.
with nlp_test.select_pipes(disable=["ner", "entity_linker"]):
    doc = nlp_test(test_text)

start_char, end_char = 49, 53 #17, 43
span = doc.char_span(start_char, end_char, label="PER", alignment_mode="strict")

if span is None:
    print("⚠️ Span non alignable, ajustez les offsets.")
else:
    doc.ents = [span]
    # Maintenant qu'on a doc.sents correctement rempli (via parser/senter),
    # on peut appeler l'entity_linker directement.
    doc = nlp_test.get_pipe("entity_linker")(doc)
    for ent in doc.ents:
        print(f"{ent.text!r} → QID: {ent.kb_id_}")

## 10. Evaluation du modèle NEL en K-FOLD

Le K-fold cross-validation s'applique aussi au NEL, avec les mêmes principes généraux qu'en NER (mesurer la robustesse du modèle sur différents sous-ensembles, éviter de ne juger la performance que sur un seul split arbitraire train/test). Mais il y a des précautions spécifiques au NEL à prendre en compte, sans quoi le K-fold peut donner des résultats trompeurs.

Pourquoi le NEL nécessite plus de vigilance que le NER pour le split

- Fuite de données par page : vous avez ~4684 mentions réparties sur seulement 101 pages, avec en moyenne ~46 mentions par page partageant le même texte de contexte. Si un split K-fold aléatoire, exemple par exemple, place deux mentions de la même page dans des folds différents, le modèle "a déjà vu" une bonne partie du texte de test pendant l'entraînement (même si la mention précise diffère) — ça biaiserait l'évaluation à la hausse. Il faut découper par page (task_id), pas par ligne.

- Couverture de la KB entre folds : contrairement au NER où les labels (PER, SPK...) sont peu nombreux et présents partout, en NEL chaque entité (QID) peut n'apparaître que dans une poignée d'exemples. Si un QID donné n'apparaît que dans le fold de test et jamais dans le fold d'entraînement, le modèle n'a aucune chance de le prédire correctement — ce n'est pas un vrai échec du modèle, mais un artefact du split. Il faut vérifier/contrôler que les entités importantes sont représentées dans chaque fold d'entraînement.

- Le cas NIL : la proportion de NIL doit rester représentative dans chaque fold, sinon un fold pourrait sur- ou sous-représenter les cas "aucune entité connue", faussant le score de rappel/précision spécifiquement sur NIL.

Implémentation : K-fold groupé par page

In [71]:
!pip install scikit-learn

In [72]:
from sklearn.model_selection import GroupKFold
from spacy.scorer import Scorer
from spacy.training import Example
from spacy.util import minibatch
from tqdm.notebook import tqdm
import numpy as np
import random
import time
import pandas as pd

# ============================================================
# 1. CRÉATION DES FOLDS, GROUPÉS PAR PAGE (task_id)
# ============================================================

K = 5
gkf = GroupKFold(n_splits=K)
indices = np.arange(len(train_examples))

folds = list(gkf.split(indices, groups=train_task_ids))

print(f"{K} folds créés, groupés par page (task_id).")
for fold_i, (tr_idx, te_idx) in enumerate(folds):
    tr_pages = set(train_task_ids[i] for i in tr_idx)
    te_pages = set(train_task_ids[i] for i in te_idx)
    print(f"  Fold {fold_i}: {len(tr_idx)} exemples train / {len(te_idx)} exemples test "
          f"({len(tr_pages)} pages train / {len(te_pages)} pages test, "
          f"chevauchement pages: {len(tr_pages & te_pages)})")

# ============================================================
# 2. FONCTION D'ÉVALUATION SUR UN FOLD DE TEST
# ============================================================

def evaluate_on_examples(nlp_model, examples):
    """Fait tourner l'entity_linker actuel sur les exemples et calcule les scores."""
    scorer = Scorer()
    predicted_examples = []
    for eg in examples:
        pred_doc = nlp_model.make_doc(eg.reference.text)

        with nlp_model.select_pipes(disable=["entity_linker"]):
            pred_doc = nlp_model(pred_doc)

        pred_doc.ents = [
            pred_doc.char_span(ent.start_char, ent.end_char, label=ent.label_, alignment_mode="strict")
            for ent in eg.reference.ents
        ]
        pred_doc.ents = [e for e in pred_doc.ents if e is not None]

        pred_doc = nlp_model.get_pipe("entity_linker")(pred_doc)

        eg_scored = Example(pred_doc, eg.reference)
        predicted_examples.append(eg_scored)

    return scorer.score_links(predicted_examples, negative_labels=["NIL"])

# ============================================================
# 3. BOUCLE K-FOLD : ENTRAINEMENT + ÉVALUATION PAR FOLD
# ============================================================

N_ITER_FOLD = 20
BATCH_SIZE = 32
EVAL_EVERY = 5

# Dossier de sortie dédié aux modèles par fold (pour l'analyse d'erreurs après coup)
FOLD_MODELS_DIR = NEL_MODEL_OUTPUT_DIR + "kfold_models/"
import os
os.makedirs(FOLD_MODELS_DIR, exist_ok=True)

fold_results = []
fold_histories = []
fold_test_examples_by_fold = {}   # <-- NOUVEAU : garder les exemples de test de chaque fold, alignés avec les modèles
nlp_fold_trained = {}             # <-- NOUVEAU : garder les modèles entraînés en mémoire

for fold_i, (tr_idx, te_idx) in enumerate(folds):
    print(f"\n=== Fold {fold_i+1}/{K} ===")
    t_fold_start = time.time()

    fold_train_examples = [train_examples[i] for i in tr_idx]
    fold_test_examples = [train_examples[i] for i in te_idx]

    nlp_fold = spacy.load(SPACY_MODEL, exclude=["entity_linker", "lemmatizer"])
    entity_linker_fold = nlp_fold.add_pipe("entity_linker", last=True)
    entity_linker_fold.set_kb(lambda vocab: spacy_kb)

    optimizer = nlp_fold.initialize(get_examples=lambda: fold_train_examples)

    fold_history = []
    for i in range(N_ITER_FOLD):
        random.shuffle(fold_train_examples)
        losses = {}
        for batch in minibatch(fold_train_examples, size=BATCH_SIZE):
            nlp_fold.update(batch, sgd=optimizer, losses=losses)

        elapsed = time.time() - t_fold_start
        log_line = f"  Fold {fold_i+1} | Iter {i+1:2d}/{N_ITER_FOLD} | loss={losses.get('entity_linker', 0):.4f} | {elapsed/60:.1f} min"

        if (i + 1) % EVAL_EVERY == 0 or (i + 1) == N_ITER_FOLD:
            scores = evaluate_on_examples(nlp_fold, fold_test_examples)
            log_line += f" | F={scores.get('nel_micro_f', 0):.3f} P={scores.get('nel_micro_p', 0):.3f} R={scores.get('nel_micro_r', 0):.3f}"
            fold_history.append({"fold": fold_i+1, "iter": i+1, "loss": losses.get('entity_linker', 0), **scores})

        print(log_line)

    fold_histories.extend(fold_history)

    final_scores = fold_history[-1]
    fold_results.append(final_scores)
    print(f"  ✅ Fold {fold_i+1} terminé en {(time.time()-t_fold_start)/60:.1f} min | F final = {final_scores.get('nel_micro_f', 0):.3f}")

    # --- NOUVEAU : sauvegarde du modèle du fold, en mémoire et sur disque ---
    nlp_fold_trained[fold_i] = nlp_fold
    fold_test_examples_by_fold[fold_i] = fold_test_examples

    fold_model_path = f"{FOLD_MODELS_DIR}fold_{fold_i}_model"
    nlp_fold.to_disk(fold_model_path)
    print(f"  💾 Modèle sauvegardé : {fold_model_path}")

# ============================================================
# 4. RÉSUMÉ FINAL SUR TOUS LES FOLDS
# ============================================================

results_df = pd.DataFrame(fold_results)
print("\n=== Résumé K-fold ===")
print(results_df[["fold", "nel_micro_p", "nel_micro_r", "nel_micro_f"]])
print("\nMoyenne :")
print(results_df[["nel_micro_p", "nel_micro_r", "nel_micro_f"]].mean())
print("\nÉcart-type :")
print(results_df[["nel_micro_p", "nel_micro_r", "nel_micro_f"]].std())

history_df = pd.DataFrame(fold_histories)

5 folds créés, groupés par page (task_id).
  Fold 0: 3739 exemples train / 936 exemples test (87 pages train / 14 pages test, chevauchement pages: 0)
  Fold 1: 3739 exemples train / 936 exemples test (81 pages train / 20 pages test, chevauchement pages: 0)
  Fold 2: 3743 exemples train / 932 exemples test (79 pages train / 22 pages test, chevauchement pages: 0)
  Fold 3: 3740 exemples train / 935 exemples test (78 pages train / 23 pages test, chevauchement pages: 0)
  Fold 4: 3739 exemples train / 936 exemples test (79 pages train / 22 pages test, chevauchement pages: 0)

=== Fold 1/5 ===
  Fold 1 | Iter  1/20 | loss=37.9594 | 1.1 min
  Fold 1 | Iter  2/20 | loss=30.4996 | 2.2 min
  Fold 1 | Iter  3/20 | loss=27.9646 | 3.2 min
  Fold 1 | Iter  4/20 | loss=26.1966 | 4.2 min


/home/cbrando/nel-env/lib/python3.13/site-packages/spacy/pipeline/attributeruler.py:138: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  matches = self.matcher(doc, allow_missing=True, as_spans=False)


  Fold 1 | Iter  5/20 | loss=24.8991 | 5.2 min | F=0.552 P=0.863 R=0.406
  Fold 1 | Iter  6/20 | loss=23.8386 | 6.4 min
  Fold 1 | Iter  7/20 | loss=22.9141 | 7.4 min
  Fold 1 | Iter  8/20 | loss=22.2621 | 8.4 min
  Fold 1 | Iter  9/20 | loss=21.6734 | 9.4 min
  Fold 1 | Iter 10/20 | loss=20.9270 | 10.4 min | F=0.560 P=0.875 R=0.412
  Fold 1 | Iter 11/20 | loss=20.4659 | 11.7 min
  Fold 1 | Iter 12/20 | loss=19.9179 | 12.7 min
  Fold 1 | Iter 13/20 | loss=19.6055 | 13.8 min
  Fold 1 | Iter 14/20 | loss=19.1788 | 14.8 min
  Fold 1 | Iter 15/20 | loss=18.8178 | 15.8 min | F=0.563 P=0.879 R=0.414
  Fold 1 | Iter 16/20 | loss=18.3571 | 17.1 min
  Fold 1 | Iter 17/20 | loss=18.0634 | 18.1 min
  Fold 1 | Iter 18/20 | loss=17.5870 | 19.2 min
  Fold 1 | Iter 19/20 | loss=17.3270 | 20.3 min
  Fold 1 | Iter 20/20 | loss=17.0778 | 21.4 min | F=0.563 P=0.879 R=0.414
  ✅ Fold 1 terminé en 21.6 min | F final = 0.563
  💾 Modèle sauvegardé : ../models/kfold_models/fold_0_model

=== Fold 2/5 ===
  Fold

'''
=== Résumé K-fold ===
   fold  nel_micro_p  nel_micro_r  nel_micro_f
0     1     0.865429     0.407205     0.553823
1     2     0.894040     0.470383     0.616438
2     3     0.942105     0.394708     0.556333
3     4     0.913043     0.454545     0.606936
4     5     0.907193     0.436872     0.589744

Moyenne :
nel_micro_p    0.904362
nel_micro_r    0.432743
nel_micro_f    0.584655
dtype: float64

Écart-type :
nel_micro_p    0.027974
nel_micro_r    0.031655
nel_micro_f    0.028659
'''

### Error analysis

In [73]:
def collect_predictions(nlp_model, examples, fold_id=None):
    """
    Comme evaluate_on_examples, mais retourne le détail par exemple :
    texte, mention, entité prédite, entité attendue, correct ou non.
    """
    rows = []
    for eg in examples:
        pred_doc = nlp_model.make_doc(eg.reference.text)

        with nlp_model.select_pipes(disable=["entity_linker"]):
            pred_doc = nlp_model(pred_doc)

        pred_doc.ents = [
            pred_doc.char_span(ent.start_char, ent.end_char, label=ent.label_, alignment_mode="strict")
            for ent in eg.reference.ents
        ]
        pred_doc.ents = [e for e in pred_doc.ents if e is not None]

        pred_doc = nlp_model.get_pipe("entity_linker")(pred_doc)

        # Associer chaque entité prédite à son entité de référence correspondante (même span)
        gold_by_span = {(e.start_char, e.end_char): e.kb_id_ for e in eg.reference.ents}

        for ent in pred_doc.ents:
            gold_kb_id = gold_by_span.get((ent.start_char, ent.end_char))
            rows.append({
                "fold": fold_id,
                "text_context": eg.reference.text[max(0, ent.start_char-80):ent.end_char+80],
                "mention": ent.text,
                "predicted_qid": ent.kb_id_,
                "gold_qid": gold_kb_id,
                "correct": ent.kb_id_ == gold_kb_id,
                "gold_is_nil": gold_kb_id == "NIL",
                "predicted_is_nil": ent.kb_id_ == "NIL",
            })
    return rows
    
all_predictions = []
for fold_i in range(K):
    preds = collect_predictions(
        nlp_fold_trained[fold_i],
        fold_test_examples_by_fold[fold_i],
        fold_id=fold_i + 1
    )
    all_predictions.extend(preds)

pred_df = pd.DataFrame(all_predictions)
print(f"{len(pred_df)} prédictions collectées sur l'ensemble des folds de test.")

/home/cbrando/nel-env/lib/python3.13/site-packages/spacy/pipeline/attributeruler.py:138: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  matches = self.matcher(doc, allow_missing=True, as_spans=False)


4675 prédictions collectées sur l'ensemble des folds de test.


Vue d'ensemble : taux d'erreur global et par catégorie

In [74]:
print("Taux de correction global :", pred_df["correct"].mean())

print("\nRépartition des erreurs par type :")
print(pd.crosstab(pred_df["gold_is_nil"], pred_df["predicted_is_nil"],
                   rownames=["Gold = NIL"], colnames=["Prédit = NIL"]))

Taux de correction global : 0.4504812834224599

Répartition des erreurs par type :
Prédit = NIL  False  True 
Gold = NIL                
False          2134   2369
True             21    151


Ce tableau croisé révèle les 4 cas fondamentaux :
- Gold NIL / Prédit NIL : vrai négatif (bien géré).
- Gold NIL / Prédit ≠ NIL : faux positif — le modèle invente un lien alors qu'il n'y en a pas.
- Gold ≠ NIL / Prédit NIL : faux négatif — le modèle "abandonne" alors qu'un lien existait.
- Gold ≠ NIL / Prédit ≠ NIL mais différent : confusion entre deux entités réelles.

Isoler et lire les erreurs concrètes, par catégorie

In [75]:
# Faux positifs : le modèle invente un lien
false_positives = pred_df[(pred_df["gold_is_nil"]) & (~pred_df["predicted_is_nil"])]
print(f"\n{len(false_positives)} faux positifs (NIL prédit comme une entité) :")
print(false_positives[["mention", "predicted_qid", "text_context"]].head(10))

# Faux négatifs : le modèle rate une entité connue
false_negatives = pred_df[(~pred_df["gold_is_nil"]) & (pred_df["predicted_is_nil"])]
print(f"\n{len(false_negatives)} faux négatifs (entité connue prédite NIL) :")
print(false_negatives[["mention", "gold_qid", "text_context"]].head(10))

# Confusions entre deux entités réelles (souvent le cas le plus informatif)
confusions = pred_df[(~pred_df["correct"]) & (~pred_df["gold_is_nil"]) & (~pred_df["predicted_is_nil"])]
print(f"\n{len(confusions)} confusions entre entités connues :")
print(confusions[["mention", "gold_qid", "predicted_qid", "text_context"]].head(10))


21 faux positifs (NIL prédit comme une entité) :
          mention predicted_qid  \
733   Emile Faure     Q13091123   
735   Emile Faure     Q13091123   
817   Bouthillier      Q2669405   
818         Bizot      Q3059731   
821          Brun     Q15973958   
1027       Rouger        Q29767   
1178  Emile Faure     Q13091123   
1249       Pascal      Q3271603   
1250        Girod     Q16025783   
2094      Boucher      Q3288708   

                                           text_context  
733   t, vous donner le renseignement qu'on sollicit...  
735   ont été les coefficients d'augmentation du pri...  
817    des comptes;\n\nM. Weil-Raynal, attaché au ca...  
818   at en service extraordinaire, directeur du bud...  
821    des contributions directes.\n\nPour assister ...  
1027  sdisant blessés désignés par quelques journaux...  
1178  nant notamment au train des équipages et aux s...  
1249  risonniers de guerre, pour lesquels, d'ailleur...  
1250   de guerre, pour lesquels, d'ailleur

Repérer les mentions/alias les plus problématiques

In [76]:
error_by_mention = (
    pred_df.groupby("mention")
    .agg(n_total=("correct", "size"), n_errors=("correct", lambda x: (~x).sum()))
    .assign(error_rate=lambda d: d["n_errors"] / d["n_total"])
    .sort_values("n_errors", ascending=False)
)
print("\nMentions les plus problématiques :")
print(error_by_mention.head(15))


Mentions les plus problématiques :
                                       n_total  n_errors  error_rate
mention                                                             
le président                               557       557         1.0
le ministre des finances                   238       238         1.0
le ministre                                165       165         1.0
le ministre des pensions                   105       105         1.0
le rapporteur                               75        75         1.0
Jean Desbons                                65        65         1.0
de Lamarzelle                               56        56         1.0
le ministre de l'instruction publique       30        30         1.0
le général Robert                           30        30         1.0
du Breuil de Saint-Germain                  25        25         1.0
Armand Chouffet                             25        25         1.0
le ministre de la guerre                    24        24         1.

Repérer les entités (QID) les plus mal reconnues côté gold

In [77]:
error_by_gold_qid = (
    pred_df[~pred_df["gold_is_nil"]]
    .groupby("gold_qid")
    .agg(n_total=("correct", "size"), n_errors=("correct", lambda x: (~x).sum()))
    .assign(error_rate=lambda d: d["n_errors"] / d["n_total"])
    .sort_values(["n_errors", "error_rate"], ascending=False)
)
print("\nEntités (QID) les plus souvent mal liées :")
print(error_by_gold_qid.head(15))


Entités (QID) les plus souvent mal liées :
          n_total  n_errors  error_rate
gold_qid                               
Q3132698      256       248    0.968750
Q459510       133       133    1.000000
Q2978569      127       127    1.000000
Q296673       123       123    1.000000
Q3143189      121       121    1.000000
Q471311        92        92    1.000000
Q585025        70        70    1.000000
Q3171692      121        65    0.537190
Q3121364       58        58    1.000000
Q3184755       59        58    0.983051
Q1934639       48        47    0.979167
Q3188688       39        39    1.000000
Q2899637       45        38    0.844444
Q317957        43        37    0.860465
Q158772        66        36    0.545455
